# 🎬 CineCut - Compilador Automático no Google Colab

[![Abrir no Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Miraplay2025/Studio-editor-automatico-/blob/main/CineCut_Build_Notebook.ipynb)

Este notebook clona automaticamente o repositório **[Miraplay2025/Studio-editor-automatico-](https://github.com/Miraplay2025/Studio-editor-automatico-.git)**, prepara o ambiente Android (Java 17 / Gradle) e compila o APK do **CineCut** sem poluição de logs no terminal, baixando o APK gerado diretamente no final da execução.

In [ ]:
# @title 🚀 Clique no botão Play para Compilar e Baixar o APK do CineCut { display-mode: "form" }
# @markdown Pressione o botão Play à esquerda (ou Ctrl+F9) para rodar o processo automaticamente.

import os
import sys
import subprocess
from IPython.display import HTML, display, clear_output

REPO_URL = "https://github.com/Miraplay2025/Studio-editor-automatico-.git"
REPO_DIR = "Studio-editor-automatico-"

def print_status(message, color="#7C5CFC", icon="⏳"):
    display(HTML(f"""
    <div style="padding:12px 18px; background-color:#161824; border-left:5px solid {color}; border-radius:8px; font-family:'Segoe UI',sans-serif; margin:10px 0;">
        <span style="font-size:18px;">{icon}</span> 
        <strong style="color:#FFFFFF; font-size:15px; margin-left:8px;">{message}</strong>
    </div>
    """))

def print_error(error_text):
    display(HTML(f"""
    <div style="padding:15px; background-color:#2D151A; border:1px solid #FF3366; border-radius:8px; font-family:monospace; color:#FF99AA; margin:10px 0;">
        <h4 style="color:#FF3366; margin:0 0 8px 0;">❌ ERRO DURANTE A COMPILAÇÃO:</h4>
        <pre style="white-space: pre-wrap; word-break: break-all; margin:0;">{error_text}</pre>
    </div>
    """))

def run_quiet_command(cmd, step_name):
    print_status(f"{step_name}...", "#00E5FF", "⚙️")
    process = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    stdout, stderr = process.communicate()
    if process.returncode != 0:
        print_error(f"Falha no passo: {step_name}\n\n{stderr if stderr else stdout}")
        sys.exit(1)
    return stdout

clear_output()
print_status("Iniciando o ambiente de compilação do CineCut...", "#7C5CFC", "🚀")

# 1. Clonar ou atualizar o repositório
if not os.path.exists(REPO_DIR):
    run_quiet_command(f"git clone {REPO_URL}", "Clonando repositório Miraplay2025/Studio-editor-automatico-")
else:
    run_quiet_command(f"cd {REPO_DIR} && git pull", "Atualizando repositório existente")

os.chdir(os.path.join(os.getcwd(), REPO_DIR))

# 2. Instalação e Configuração do Java JDK 17
run_quiet_command("apt-get update -qq && apt-get install -y -qq openjdk-17-jdk > /dev/null 2>&1", "Instalando OpenJDK 17")
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

# 3. Permissões do Gradle Wrapper
if os.path.exists("gradlew"):
    os.chmod("gradlew", 0o755)
else:
    run_quiet_command("apt-get install -y -qq gradle > /dev/null 2>&1", "Instalando Gradle de fallback")

# 4. Aceitar Licenças do Android SDK
run_quiet_command('yes | sdkmanager --licenses > /dev/null 2>&1 || true', "Configurando Licenças do Android SDK")

# 5. Compilação do APK
print_status("Compilando o APK (isso leva aproximadamente 1-3 minutos)...", "#7C5CFC", "🔨")
build_cmd = "./gradlew assembleDebug --quiet" if os.path.exists("gradlew") else "gradle assembleDebug --quiet"
run_quiet_command(build_cmd, "Compilando APK do CineCut")

# 6. Localizar o APK Gerado
apk_path = None
for root, dirs, files in os.walk("app/build/outputs/apk"):
    for file in files:
        if file.endswith(".apk"):
            apk_path = os.path.join(root, file)
            break

if not apk_path or not os.path.exists(apk_path):
    print_error("Não foi possível localizar o APK em app/build/outputs/apk/")
    sys.exit(1)

print_status("APK compilado com sucesso! Iniciando download...", "#00E5FF", "✅")

# 7. Download Automático no Colab
from google.colab import files
display(HTML(f"""
<div style="background: linear-gradient(135deg, #161824 0%, #202334 100%); padding:25px; border-radius:12px; border: 2px solid #7C5CFC; text-align:center; margin-top:15px;">
    <h2 style="color:#FFFFFF; margin-top:0;">🎉 Compilação Concluída com Sucesso!</h2>
    <p style="color:#A0A5C0; font-size:14px;">O download do arquivo <strong>{os.path.basename(apk_path)}</strong> do repositório <code>Miraplay2025/Studio-editor-automatico-</code> iniciará automaticamente.</p>
</div>
"""))
files.download(apk_path)
